# TF-IDF + pca + Gradient Boosted Trees

Pipeline:
1. Stratified 30% sample of train (preserves class proportions)
2. Compute + attach class weights to sample
3. Fit TF-IDF on sample → frozen transformer
4. Apply TF-IDF to all 3 splits → persist
5. Fit pca on persisted train sample TF-IDF → frozen transformer
6. Apply pca to all 3 splits → persist
7. Join pca features onto original splits, assemble with structural features → checkpoint
8. Fit GBTClassifier on checkpointed train

## 0. Keepalive Thread

In [1]:
import time, threading

def keepalive():
    while True:
        time.sleep(300)
        print("keepalive ping", flush=True)

t = threading.Thread(target=keepalive, daemon=True)
t.start()
print("Keepalive thread started.")

Keepalive thread started.


## 1. Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("TFIDF_pca_GBT") \
    .master("local[*]") \
    .config("spark.driver.memory", "120g") \
    .config("spark.driver.maxResultSize", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.local.dir", "/expanse/lustre/scratch/jkeeton/temp_project/spark-tmp") \
    .getOrCreate()

spark.sparkContext.setCheckpointDir("/expanse/lustre/scratch/jkeeton/temp_project/spark-checkpoints")
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Spark version: 3.5.0


## 2. Load Train / Val / Test Splits

In [3]:
TRAIN_DIR = "/home/jkeeton/ekim18/reddit-engagement-classifier/data/processed/train_model_v2"
VAL_DIR   = "/home/jkeeton/ekim18/reddit-engagement-classifier/data/processed/val_model_v2"
TEST_DIR  = "/home/jkeeton/ekim18/reddit-engagement-classifier/data/processed/test_model_v2"

train_df = spark.read.parquet(TRAIN_DIR)
val_df   = spark.read.parquet(VAL_DIR)
test_df  = spark.read.parquet(TEST_DIR)

print(f"Train rows: {train_df.count():,}")
print(f"Val rows:   {val_df.count():,}")
print(f"Test rows:  {test_df.count():,}")

Train rows: 276,442,594
Val rows:   114,089,205
Test rows:  144,949,019


## 3. Structural Feature Columns

In [4]:
STRUCTURAL_COLS = [
    "title_len", "has_question", "has_exclamation",
    "title_has_number", "title_is_allcaps",
    "has_title", "is_text_post", "selftext_len",
    "hour_of_day", "day_of_week",
    "is_known_bot", "is_anonymous_author",
    "author_post_count", "author_mean_score",
    "subreddit_post_count", "subreddit_median_score", "subreddit_median_comments",
    "title_sentiment"
]
print(f"Structural feature count: {len(STRUCTURAL_COLS)}")

Structural feature count: 18


## 4. Stratified 30% Train Sample

Matches the XGBoost arm's M3 v2 sample exactly — same fraction, same seed, same label column.

In [5]:
SAMPLE_FRAC = 0.30
fractions = {0.0: SAMPLE_FRAC, 1.0: SAMPLE_FRAC, 2.0: SAMPLE_FRAC, 3.0: SAMPLE_FRAC}

train_sample = train_df.sampleBy("label_4class_idx", fractions=fractions, seed=42)

print(f"Sampled train rows: {train_sample.count():,}")
print("Class distribution in sample:")
train_sample.groupBy("label_4class_idx").count().orderBy("label_4class_idx").show()

Sampled train rows: 82,930,798
Class distribution in sample:
+----------------+--------+
|label_4class_idx|   count|
+----------------+--------+
|             0.0|39105582|
|             1.0|21306558|
|             2.0|12266449|
|             3.0|10252209|
+----------------+--------+



## 5. Compute and Attach Class Weights to Sample

Weights are computed on the sampled distribution (post-stratification) and attached as a column for use in GBT training.

In [6]:
from itertools import chain

def compute_class_weights(df, label_col):
    counts = df.groupBy(label_col).count().collect()
    total  = sum(r["count"] for r in counts)
    n_cls  = len(counts)
    return {int(r[label_col]): total / (n_cls * r["count"]) for r in counts}

w4 = compute_class_weights(train_sample, "label_4class_idx")
print("4-class weights:", {k: round(v, 4) for k, v in sorted(w4.items())})

map4 = F.create_map([F.lit(x) for x in chain(*w4.items())])
train_sample = train_sample.withColumn("weight_4class", map4[F.col("label_4class_idx")])
print("Weight column attached to train_sample.")

4-class weights: {0: 0.5302, 1: 0.9731, 2: 1.6902, 3: 2.0223}
Weight column attached to train_sample.


## 6. Fit TF-IDF on Train Sample

2000 HashingTF buckets gives pca enough signal. `minDocFreq=5` drops very rare terms. Fit only — transform happens in the next cell.

In [7]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

tokenizer  = Tokenizer(inputCol="title", outputCol="title_tokens")
remover    = StopWordsRemover(inputCol="title_tokens", outputCol="title_tokens_clean")
hashing_tf = HashingTF(inputCol="title_tokens_clean", outputCol="title_tf", numFeatures=2000)
idf        = IDF(inputCol="title_tf", outputCol="title_tfidf_raw", minDocFreq=5)

tfidf_pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf])

print("Fitting TF-IDF on 30% stratified train sample...")
t0 = time.time()
tfidf_model = tfidf_pipeline.fit(train_sample)
print(f"TF-IDF fit time: {time.time() - t0:.1f}s")

Fitting TF-IDF on 30% stratified train sample...
TF-IDF fit time: 116.8s


## 7. Apply TF-IDF to All Splits → Persist

Persist immediately after transform to break lineage before pca fit.

In [8]:
TFIDF_TRAIN_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/train_tfidf"
TFIDF_VAL_DIR   = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/val_tfidf"
TFIDF_TEST_DIR  = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/test_tfidf"

print("Transforming and persisting TF-IDF for all splits...")
t0 = time.time()

for df, path, name in [
    (train_df, TFIDF_TRAIN_DIR, "train"),
    (val_df,   TFIDF_VAL_DIR,   "val"),
    (test_df,  TFIDF_TEST_DIR,  "test"),
]:
    tfidf_model.transform(df) \
        .select("id", "title_tfidf_raw") \
        .write.mode("overwrite").parquet(path)
    print(f"  {name} done → {path}")

print(f"Total TF-IDF persist time: {time.time() - t0:.1f}s")

Transforming and persisting TF-IDF for all splits...
keepalive ping
keepalive ping
  train done → /expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/train_tfidf
  val done → /expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/val_tfidf
keepalive ping
  test done → /expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/test_tfidf
Total TF-IDF persist time: 857.8s


In [9]:
# ── RESUME POINT: TF-IDF persisted, restart from here if session died ─────────
# Requires: Spark session (§1), STRUCTURAL_COLS (§3), path constants (§7)
# Re-derive sample_ids and map4 for use in later sections.

from itertools import chain
from pyspark.sql import functions as F

TFIDF_TRAIN_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/train_tfidf"
TFIDF_VAL_DIR   = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/val_tfidf"
TFIDF_TEST_DIR  = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/test_tfidf"

train_tfidf_full = spark.read.parquet(TFIDF_TRAIN_DIR)
val_tfidf        = spark.read.parquet(TFIDF_VAL_DIR)
test_tfidf       = spark.read.parquet(TFIDF_TEST_DIR)

print(f"Reloaded TF-IDF splits:")
print(f"  train: {train_tfidf_full.count():,} rows")
print(f"  val:   {val_tfidf.count():,} rows")
print(f"  test:  {test_tfidf.count():,} rows")

# Re-derive stratified sample IDs so PCA fit cell (§8) can filter correctly
SAMPLE_FRAC = 0.30
fractions   = {0.0: SAMPLE_FRAC, 1.0: SAMPLE_FRAC, 2.0: SAMPLE_FRAC, 3.0: SAMPLE_FRAC}

train_df    = spark.read.parquet("/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/train")
train_sample = train_df.sampleBy("label_4class_idx", fractions=fractions, seed=42)
sample_ids   = train_sample.select("id")

# Re-derive class weight map
def compute_class_weights(df, label_col):
    counts = df.groupBy(label_col).count().collect()
    total  = sum(r["count"] for r in counts)
    n_cls  = len(counts)
    return {int(r[label_col]): total / (n_cls * r["count"]) for r in counts}

w4   = compute_class_weights(train_sample, "label_4class_idx")
map4 = F.create_map([F.lit(x) for x in chain(*w4.items())])
print("4-class weights:", {k: round(v, 4) for k, v in sorted(w4.items())})
print("sample_ids and map4 ready.")

Reloaded TF-IDF splits:
  train: 276,442,594 rows
  val:   114,089,205 rows
  test:  144,949,019 rows
4-class weights: {0: 0.5301, 1: 0.9731, 2: 1.6899, 3: 2.0231}
sample_ids and map4 ready.


## 8. Fit pca on Train Sample TF-IDF

Reload the persisted train TF-IDF and filter to the same stratified sample rows via `id` join. `k=100` components.

In [10]:
from pyspark.ml.feature import PCA

pca_K = 100

# Get sample IDs to filter persisted TF-IDF to same rows used for TF-IDF fit
sample_ids = train_sample.select("id")
train_tfidf_full   = spark.read.parquet(TFIDF_TRAIN_DIR)
train_tfidf_sample = train_tfidf_full.join(sample_ids, on="id")

print(f"Train TF-IDF sample rows for pca fit: {train_tfidf_sample.count():,}")

pca = PCA(k=pca_K, inputCol="title_tfidf_raw", outputCol="title_pca")

print(f"Fitting pca (k={pca_K}) on train sample TF-IDF...")
t0 = time.time()
pca_model = pca.fit(train_tfidf_sample)
print(f"pca fit time: {time.time() - t0:.1f}s")

explained = float(pca_model.explainedVariance.toArray().sum())
print(f"Cumulative explained variance (k={pca_K}): {explained:.3f}")

keepalive ping
Train TF-IDF sample rows for pca fit: 82,925,706
Fitting pca (k=100) on train sample TF-IDF...
keepalive ping
pca fit time: 194.6s
Cumulative explained variance (k=100): 0.132


## 9. Apply pca to All Splits → Persist

Final NLP feature parquets. This is the last persist before assembly — clean checkpoint boundary.

In [11]:
pca_TRAIN_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/train_pca"
pca_VAL_DIR   = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/val_pca"
pca_TEST_DIR  = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/test_pca"

print("Transforming and persisting pca for all splits...")
t0 = time.time()

for tfidf_path, pca_path, name in [
    (TFIDF_TRAIN_DIR, pca_TRAIN_DIR, "train"),
    (TFIDF_VAL_DIR,   pca_VAL_DIR,   "val"),
    (TFIDF_TEST_DIR,  pca_TEST_DIR,  "test"),
]:
    tfidf_df = spark.read.parquet(tfidf_path)
    pca_model.transform(tfidf_df) \
        .select("id", "title_pca") \
        .write.mode("overwrite").parquet(pca_path)
    print(f"  {name} done → {pca_path}")

print(f"Total pca persist time: {time.time() - t0:.1f}s")

Transforming and persisting pca for all splits...
keepalive ping
  train done → /expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/train_pca
  val done → /expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/val_pca
keepalive ping
  test done → /expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/test_pca
Total pca persist time: 668.8s


In [4]:
# # ── RESUME POINT: PCA persisted, restart from here if session died ────────────
# # Requires: Spark session (§1), STRUCTURAL_COLS (§3), map4 + sample_ids (§7a)
# # If starting fresh here, run §7a first to restore map4 and sample_ids.

# from pyspark.ml.feature import VectorAssembler

# PCA_TRAIN_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/train_pca"
# PCA_VAL_DIR   = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/val_pca"
# PCA_TEST_DIR  = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/test_pca"

# train_pca = spark.read.parquet(PCA_TRAIN_DIR)
# val_pca   = spark.read.parquet(PCA_VAL_DIR)
# test_pca  = spark.read.parquet(PCA_TEST_DIR)

# print(f"Reloaded PCA splits:")
# print(f"  train: {train_pca.count():,} rows")
# print(f"  val:   {val_pca.count():,} rows")
# print(f"  test:  {test_pca.count():,} rows")

# # Reload original splits if not already in memory
# train_df = spark.read.parquet("/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/train")
# val_df   = spark.read.parquet("/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/val")
# test_df  = spark.read.parquet("/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/test")

# # Assemble combined vectors
# assembler = VectorAssembler(
#     inputCols=["features", "title_pca"],
#     outputCol="features_combined",
#     handleInvalid="skip"
# )

# def load_and_assemble(base_df, pca_path):
#     pca_df = spark.read.parquet(pca_path)
#     return assembler.transform(base_df.join(pca_df, on="id"))

# train_combined = load_and_assemble(train_df, PCA_TRAIN_DIR)
# val_combined   = load_and_assemble(val_df,   PCA_VAL_DIR)
# test_combined  = load_and_assemble(test_df,  PCA_TEST_DIR)

# sample_vec = train_combined.select("features_combined").first()[0]
# print(f"Combined feature vector size: {len(sample_vec)}")

# # Re-derive sample_ids and map4
# from itertools import chain

# SAMPLE_FRAC = 0.30
# fractions   = {0.0: SAMPLE_FRAC, 1.0: SAMPLE_FRAC, 2.0: SAMPLE_FRAC, 3.0: SAMPLE_FRAC}
# train_sample = train_df.sampleBy("label_4class_idx", fractions=fractions, seed=42)
# sample_ids   = train_sample.select("id")

# def compute_class_weights(df, label_col):
#     counts = df.groupBy(label_col).count().collect()
#     total  = sum(r["count"] for r in counts)
#     n_cls  = len(counts)
#     return {int(r[label_col]): total / (n_cls * r["count"]) for r in counts}

# w4   = compute_class_weights(train_sample, "label_4class_idx")
# map4 = F.create_map([F.lit(x) for x in chain(*w4.items())])
# print("4-class weights:", {k: round(v, 4) for k, v in sorted(w4.items())})
# print("sample_ids and map4 ready.")
# print("train/val/test_combined ready for §11.")

Reloaded PCA splits:
  train: 276,442,594 rows
  val:   114,089,205 rows
  test:  144,949,019 rows
keepalive ping
keepalive ping
Combined feature vector size: 118
train/val/test_combined ready for §11.


## 10. Join pca Features, Assemble Combined Vector

Join pca parquets back onto original splits, then assemble structural + pca into one vector.

`features` = pre-scaled structural vector from Milestone 3 (18 dims)
`title_pca` = pca-compressed TF-IDF (100 dims)
→ **118 combined features (dense)**

In [6]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["features", "title_pca"],
    outputCol="features_combined",
    handleInvalid="skip"
)

PCA_TRAIN_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/train_pca"
PCA_VAL_DIR   = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/val_pca"
PCA_TEST_DIR  = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/processed/tfidf_pca/test_pca"

def load_and_assemble(base_df, pca_path):
    pca_df = spark.read.parquet(pca_path)
    joined = base_df.join(pca_df, on="id")
    return assembler.transform(joined)

train_combined = load_and_assemble(train_df, PCA_TRAIN_DIR)
val_combined   = load_and_assemble(val_df,   PCA_VAL_DIR)
test_combined  = load_and_assemble(test_df,  PCA_TEST_DIR)

sample_vec = train_combined.select("features_combined").first()[0]
print(f"Combined feature vector size: {len(sample_vec)}")

keepalive ping
keepalive ping
Combined feature vector size: 118


## 11. Attach Class Weights to train_combined, Then Checkpoint

Re-attach weights to the full assembled train df (sample had them, but we need them on the full train for GBT). Then checkpoint to cut lineage before fitting.

In [10]:
train_combined = train_combined.withColumn("weight_4class", map4[F.col("label_4class_idx")])

# Also filter train_combined to the stratified sample for fitting
train_combined_sample = train_combined.join(sample_ids, on="id")

In [ ]:
# train_combined_sample = train_combined_sample.checkpoint()
# print(f"Checkpoint time: {time.time() - t0:.1f}s")
# print(f"Sample rows after checkpoint: {train_combined_sample.count():,}")

In [ ]:
# # ── RESUME POINT: checkpoint written, restart from here if session died ───────
# # Requires: Spark session (§1), val_combined + test_combined (rebuild via §9a)
# # The checkpoint dir holds the materialized sample — reload it directly.

# from pyspark.ml.classification import GBTClassifier, OneVsRest
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# import time

# CHECKPOINT_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/spark-checkpoints"

# # Spark writes checkpoints as numbered subdirs; read the most recent one.
# import subprocess
# result = subprocess.run(
#     ["ls", "-t", CHECKPOINT_DIR], capture_output=True, text=True
# )
# latest = result.stdout.strip().split("\n")[0]
# checkpoint_path = f"{CHECKPOINT_DIR}/{latest}"
# print(f"Loading checkpoint from: {checkpoint_path}")

# train_combined_sample = spark.read.parquet(checkpoint_path)
# print(f"Checkpoint rows: {train_combined_sample.count():,}")
# print("Ready to run §12 (GBT fit).")

## 12. Fit GBTClassifier

Gradient Boosted Trees chosen over Random Forest:
- Sequential boosting converges in fewer trees on dense mixed features
- Better handling of class imbalance via `weightCol`
- More practical at scale: 100 trees vs 200 for comparable accuracy

`maxIter=100`, `maxDepth=6` is the standard GBT sweet spot for tabular+NLP.

In [11]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Note: PySpark GBTClassifier is binary-only natively.
# For 4-class we use One-vs-Rest wrapper.
from pyspark.ml.classification import OneVsRest

GBT_PARAMS = dict(
    maxIter=100,
    maxDepth=6,
    stepSize=0.1,        # learning rate
    subsamplingRate=0.8, # row subsampling per tree (like RF's bootstrap)
    featureSubsetStrategy="sqrt",
    seed=42
)

gbt_base = GBTClassifier(
    featuresCol="features_combined",
    labelCol="label_4class_idx",
    **GBT_PARAMS
)

from itertools import chain
from pyspark.sql import functions as F

if "weight_4class" not in train_combined_sample.columns:
    w4   = compute_class_weights(train_sample, "label_4class_idx")
    map4 = F.create_map([F.lit(x) for x in chain(*w4.items())])
    train_combined_sample = train_combined_sample.withColumn(
        "weight_4class", map4[F.col("label_4class_idx")]
    )
    print("weight_4class attached.")
else:
    print("weight_4class already present, skipping.")
    
print("weight_4class re-attached. Columns:", train_combined_sample.columns)

ovr = OneVsRest(
    classifier=gbt_base,
    featuresCol="features_combined",
    labelCol="label_4class_idx",
    weightCol="weight_4class"
)

total = train_combined_sample.count()
print(f"Current rows: {total:,}")

TARGET = 2_000_000
frac   = TARGET / total
print(f"Downsample fraction: {frac:.4f}")

train_combined_sample = train_combined_sample.sampleBy(
    "label_4class_idx",
    fractions={0.0: frac, 1.0: frac, 2.0: frac, 3.0: frac},
    seed=42
)
print(f"Downsampled rows: {train_combined_sample.count():,}")

print("Training GBT (One-vs-Rest, 4-class) on stratified 30% sample...")
print(f"Params: {GBT_PARAMS}")
t0 = time.time()
gbt_model = ovr.fit(train_combined_sample)

train_time = time.time() - t0
print(f"Training time: {train_time:.1f}s  ({train_time/60:.1f} min)")

weight_4class already present, skipping.
weight_4class re-attached. Columns: ['id', 'author', 'subreddit', 'num_comments', 'score', 'selftext', 'subreddit_id', 'title', 'created_utc', 'is_known_bot', 'is_anonymous_author', 'has_title', 'label_4class', 'label_binary', 'title_len', 'title_word_count', 'has_question', 'title_starts_question_word', 'has_exclamation', 'title_has_number', 'title_is_allcaps', 'is_text_post', 'selftext_len', 'hour_of_day', 'day_of_week', 'month', 'subreddit_post_count', 'subreddit_median_score', 'subreddit_median_comments', 'author_post_count', 'author_mean_score', 'title_sentiment', 'label_4class_idx', 'label_binary_idx', 'features_raw', 'features', 'title_pca', 'features_combined', 'weight_4class']
keepalive ping
keepalive ping
Current rows: 82,925,706
Downsample fraction: 0.0241
keepalive ping
keepalive ping
Downsampled rows: 1,998,964
Training GBT (One-vs-Rest, 4-class) on stratified 30% sample...
Params: {'maxIter': 100, 'maxDepth': 6, 'stepSize': 0.1, 's

## 13. Evaluate on Val and Test

In [12]:
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label_4class_idx",
    predictionCol="prediction",
    metricName="f1"
)

val_preds  = gbt_model.transform(val_combined)
test_preds = gbt_model.transform(test_combined)

val_f1  = evaluator_f1.evaluate(val_preds)
test_f1 = evaluator_f1.evaluate(test_preds)

print(f"Val  F1: {val_f1:.4f}")
print(f"Test F1: {test_f1:.4f}")

keepalive ping
keepalive ping
keepalive ping
keepalive ping
keepalive ping
keepalive ping
keepalive ping
keepalive ping
keepalive ping
Val  F1: 0.5761
Test F1: 0.5696


## 14. Per-Class F1 Breakdown

In [13]:
from pyspark.mllib.evaluation import MulticlassMetrics

def per_class_f1(predictions_df, label_col="label_4class_idx", pred_col="prediction"):
    rdd = predictions_df.select(pred_col, label_col) \
        .rdd.map(lambda r: (float(r[0]), float(r[1])))
    metrics = MulticlassMetrics(rdd)
    label_names = {0.0: "low-engagement", 1.0: "viral", 2.0: "crowd-pleaser", 3.0: "debate-starter"}
    return {name: round(metrics.fMeasure(lbl), 4) for lbl, name in label_names.items()}

print("Per-class F1 — Val:")
for cls, f1 in per_class_f1(val_preds).items():
    print(f"  {cls:<22} {f1:.4f}")

print("\nPer-class F1 — Test:")
for cls, f1 in per_class_f1(test_preds).items():
    print(f"  {cls:<22} {f1:.4f}")

Per-class F1 — Val:
keepalive ping


/usr/local/spark/python/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


keepalive ping
keepalive ping
  low-engagement         0.6132
  viral                  0.6930
  crowd-pleaser          0.3916
  debate-starter         0.3069

Per-class F1 — Test:
keepalive ping


/usr/local/spark/python/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


keepalive ping
keepalive ping
keepalive ping
keepalive ping
  low-engagement         0.6216
  viral                  0.6887
  crowd-pleaser          0.3184
  debate-starter         0.2999
keepalive ping
keepalive ping
keepalive ping
keepalive ping
